# DADER baseline (cross-domain transfer paradigm)

**Repo**: <https://github.com/ruc-datalab/DADER>

**Dataset code mapping** (their codes ↔ our pair names):

| Our pair | DADER --src | DADER --tgt |
|---|---|---|
| Pair #1 (WA → AB) | `wa1` | `ab` |
| Pair #2 (WDC-Co → WDC-Wa) | `computers` | `watches` |
| Pair #3 (WA → DBLP-ACM) | `wa1` | `da` |
| Pair #5 (WA → AG) | `wa1` | `ag` |



## 1. Bootstrap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = os.environ.get('REPO_ROOT') or '/content/drive/MyDrive/cd-er-paradigm-choice'
os.environ['REPO_ROOT'] = BASE
os.chdir(BASE)
print('cwd:', os.getcwd())

!git config user.email "author041@gmail.com"
!git config user.name "author97"
!git pull --ff-only || echo '[bootstrap] git pull failed; continuing'


## 2. Clone DADER + install dependencies 

One-time per Colab session. If DADER directory already exists, just installs dependencies.


In [ ]:
import subprocess, shutil
from pathlib import Path

# Install DADER ephemerally to /content (Drive FUSE drops files mid-clone).
# Re-cloned per session — DADER is ~30 MB, clones in ~10 sec.
DADER_DIR = Path('/content/DADER')

if DADER_DIR.exists():
    print(f'Removing existing {DADER_DIR}...')
    shutil.rmtree(DADER_DIR)

print(f'Cloning DADER to {DADER_DIR}...')
result = subprocess.run(
    ['git', 'clone', '--depth', '1',
     'https://github.com/ruc-datalab/DADER', str(DADER_DIR)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('Clone failed')

main_mmd = DADER_DIR / 'main' / 'main_mmd.py'
assert main_mmd.exists(), f'Clone incomplete: {main_mmd} missing'
print(f'✓ Clone OK: main_mmd.py is {main_mmd.stat().st_size} bytes')

# ────────────────────────────────────────────────────────────────────────
# CRITICAL: patch param.py IMMEDIATELY after clone. DADER's committed
# param.py is missing hidden_size (BERT-base hidden dim = 768), which
# modules/matcher.py references. Without this, training crashes with:
#   AttributeError: module 'param' has no attribute 'hidden_size'
#
# Doing this in the same cell as the clone means re-running this cell
# at any time leaves DADER in a working state. Don't separate them.
# ────────────────────────────────────────────────────────────────────────
param_file = DADER_DIR / 'param.py'
content = param_file.read_text()
patches_added = []
if 'hidden_size' not in content:
    content += '\nhidden_size = 768  # BERT-base hidden dim (patched in — missing from DADER repo)\n'
    patches_added.append('hidden_size = 768')
if 'dropout' not in content:
    content += 'dropout = 0.1  # default BERT dropout (patched in if downstream code needs it)\n'
    patches_added.append('dropout = 0.1')
if patches_added:
    param_file.write_text(content)
    print(f'✓ Patched param.py: {", ".join(patches_added)}')
else:
    print('param.py already complete (no patch needed)')

# Clean up any broken Drive copy from earlier attempts
DRIVE_DADER = Path(BASE) / 'external' / 'DADER'
if DRIVE_DADER.exists():
    print(f'Removing broken Drive copy at {DRIVE_DADER}...')
    shutil.rmtree(DRIVE_DADER)

# Install Python deps (use Colab transformers — no version pin)
!pip install -q scikit-learn pandas numpy tqdm

import transformers, torch
print(f'\ntransformers: {transformers.__version__}')
print(f'torch:        {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')


In [ ]:
# Also patch DADER's main_*.py scripts to use sys.path.insert(0, "../")
# instead of sys.path.append("../"). Their original `append` puts DADER's
# directory AFTER site-packages, so `import param` finds the PyPI
# `param` package (a parameter-validation library) instead of DADER's
# local param.py, causing AttributeError: module 'param' has no attribute 'hidden_size'.
for main_script in (DADER_DIR / 'main').glob('main_*.py'):
    content = main_script.read_text()
    if 'sys.path.append("../")' in content:
        content = content.replace('sys.path.append("../")', 'sys.path.insert(0, "../")')
        main_script.write_text(content)
        print(f'✓ Patched {main_script.name}: append → insert(0, ...)')


## 3. GPU sanity check

DADER's MMD/GRL alignment requires a forward+backward pass through 2x the batch size (source + target), so OOM risk is higher than warm-start. T4 with 16GB should be fine at batch_size=32.


In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU not attached — Runtime → Change runtime type → T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


## 4. Configure which (source, target) pair to run

The DADER paper provides multiple alignment variants; this notebook uses `main_mmd.py` (RBF-kernel MMD) by default because it aligns with our Axis A measurement framework and the paper reports it as the most stable variant.

Edit `PAIRS_TO_RUN` to select pairs. Each entry runs `main_mmd.py --src <SRC> --tgt <TGT>` and writes results to `results/runs/dader/<src>__to__<tgt>/`.


In [ ]:
# Configure pairs — comment out pairs you've already done
PAIRS_TO_RUN = [
    # ('wa1', 'ab'),         # Pair #1 — WA → AB
    ('computers', 'watches'),  # Pair #2 — WDC-Co → WDC-Wa
    ('wa1', 'da'),          # Pair #3 — WA → DBLP-ACM
    ('wa1', 'ag'),          # Pair #5 — WA → AG
]

# DADER variant — choose one. mmd is recommended for our purposes.
VARIANT = 'mmd'      # 'mmd' | 'grl' | 'invgan' | 'invgan_kd' | 'k_order' | 'ed'

# DADER's actual CLI hparam names (per main/main_mmd.py argparse):
N_EPOCH = 15         # passed as --num_epochs; DADER default = 40 (matches their paper)
BATCH_SIZE = 32      # passed as --batch_size
LAMBDA = 0.1         # passed as --beta (MMD-loss weight); DADER default = 0.1

# NOTE: learning rate is HARDCODED in DADER/param.py (c_learning_rate=2e-5,
# d_learning_rate=1e-5) and NOT exposed via CLI. To change LR, edit param.py.

SKIP_IF_DONE = True


## 5. Run the experiments

Each (src, tgt) pair takes approximately 5 GPU-hours on T4 (similar to our warm-start grid). Saves stdout to `train.log` and parses test F1 into `metrics.json`. Use the GPU-disconnect prevention trick if running multiple pairs.


In [ ]:
from pathlib import Path
patched_files = []
for main_script in Path('/content/DADER/main').glob('main_*.py'):
    content = main_script.read_text()
    if 'sys.path.append("../")' in content:
        content = content.replace('sys.path.append("../")', 'sys.path.insert(0, "../")')
        main_script.write_text(content)
        patched_files.append(main_script.name)
        print(f'✓ Patched {main_script.name}: append → insert(0, ...)')
print(f'\nPatched {len(patched_files)} file(s)')


In [ ]:
import subprocess, json, re, time, os
from pathlib import Path

DADER_DIR = Path('/content/DADER')
RUNS_DIR = Path(BASE) / 'results' / 'runs' / f'dader-{VARIANT}-ep{N_EPOCH}'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

main_script_path = DADER_DIR / 'main' / f'main_{VARIANT}.py'
assert main_script_path.exists(), f'Missing {main_script_path}'

# Run from inside DADER/main/ — DADER's code assumes this:
#   - `sys.path.append("../")` adds DADER/ to sys.path for module imports
#   - `os.path.join('../data', ...)` resolves to DADER/data/
# Their README's example also runs from main/.
RUN_CWD = DADER_DIR / 'main'

for src, tgt in PAIRS_TO_RUN:
    out_dir = RUNS_DIR / f'{src}__to__{tgt}'
    out_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = out_dir / 'metrics.json'

    if SKIP_IF_DONE and metrics_path.exists():
        existing = json.loads(metrics_path.read_text())
        if existing.get('test_f1') is not None and existing.get('returncode', 0) == 0:
            print(f'[skip] {src} → {tgt}: already done F1={existing["test_f1"]}')
            continue

    cmd = [
        'python', f'main_{VARIANT}.py',
        '--src', src, '--tgt', tgt,
        '--num_epochs', str(N_EPOCH),
        '--batch_size', str(BATCH_SIZE),
        '--beta', str(LAMBDA),
    ]
    print(f'\n=== {src} → {tgt} ({VARIANT}) ===')
    print(f'cmd: {" ".join(cmd)}')
    print(f'cwd: {RUN_CWD}')

    log_path = out_dir / 'train.log'
    start = time.time()
    with open(log_path, 'w') as log_f:
        log_f.write(f'# Command: {" ".join(cmd)}\n# cwd: {RUN_CWD}\n\n')
        proc = subprocess.run(cmd, cwd=str(RUN_CWD), capture_output=True, text=True)
        log_f.write(proc.stdout)
        log_f.write('\n--- STDERR ---\n')
        log_f.write(proc.stderr)
    elapsed = time.time() - start

    out = proc.stdout + '\n' + proc.stderr
    test_f1 = None
    m = re.search(r'=== Result of \w+:\s*===\s*\n\s*([0-9]+\.[0-9]+)', out)
    if m:
        test_f1 = float(m.group(1))
    else:
        for pattern in [
            r'(?:test[_\s]*f1|F1[_\s]*test|bestf1|best[_\s]*f1)[:\s=]+([0-9]+\.[0-9]+)',
            r'best.*?f1[^\n]*?([0-9]+\.[0-9]+)',
        ]:
            m = re.search(pattern, out, re.IGNORECASE)
            if m:
                test_f1 = float(m.group(1))
                break

    metrics = {
        'method': f'dader-{VARIANT}',
        'src': src, 'tgt': tgt,
        'beta_mmd': LAMBDA, 'num_epochs': N_EPOCH, 'batch_size': BATCH_SIZE,
        'test_f1': test_f1,
        'returncode': proc.returncode,
        'elapsed_sec': elapsed,
    }
    metrics_path.write_text(json.dumps(metrics, indent=2))
    print(f'  done in {elapsed/60:.1f} min  rc={proc.returncode}  test_f1={test_f1}')

print('\n=== Summary ===')
for src, tgt in PAIRS_TO_RUN:
    m_path = RUNS_DIR / f'{src}__to__{tgt}' / 'metrics.json'
    if m_path.exists():
        m = json.loads(m_path.read_text())
        print(f'  {src} → {tgt}: F1={m.get("test_f1")}  rc={m.get("returncode")}')


In [ ]:
log = open('$REPO_ROOT/results/runs/dader/wa1__to__ab/train.log').read()
print(f"Log size: {len(log)} bytes\n")
print("=== first 30 lines ===")
print('\n'.join(log.splitlines()[:30]))
print("\n=== last 40 lines ===")
print('\n'.join(log.splitlines()[-40:]))


In [ ]:
from pathlib import Path
import subprocess

# 1. Show current param.py contents
p = Path('/content/DADER/param.py')
print(f"=== /content/DADER/param.py (exists={p.exists()}, size={p.stat().st_size if p.exists() else 0}) ===")
if p.exists():
    print(p.read_text())

# 2. Find ALL param.py files anywhere in DADER (maybe there's another one we missed)
print("\n=== ALL param.py files anywhere in /content/DADER ===")
result = subprocess.run(['find', '/content/DADER', '-name', 'param.py', '-type', 'f'],
                        capture_output=True, text=True)
print(result.stdout or '(none found)')

# 3. Check for .pyc cache files that might be shadowing the patched .py
print("\n=== Any cached .pyc for param ===")
result = subprocess.run(['find', '/content/DADER', '-name', 'param*.pyc'],
                        capture_output=True, text=True)
print(result.stdout or '(none)')

# 4. As a last resort: forcibly patch and verify the byte-for-byte content
print("\n=== Forcing patch + verify ===")
content = p.read_text()
if 'hidden_size' not in content:
    content += '\nhidden_size = 768\n'
    p.write_text(content)
    print("Patch APPLIED")
else:
    print("hidden_size ALREADY PRESENT")

print(f"\nFinal contents:")
print(p.read_text())
print(f"\nFile mtime: {p.stat().st_mtime}")


In [ ]:
import sys
sys.path.insert(0, '/content/DADER')   # mimic the sequence main_mmd.py uses, in reverse
import param
print(f"param module file: {param.__file__}")
print(f"has hidden_size:   {hasattr(param, 'hidden_size')}")


## 6. Validation against published numbers

Their paper reports (Tables 2-3):

| Source → Target | DADER (MMD) F1 |
|---|---|
| WA → AB | 0.50–0.65 |
| WA → DBLP-ACM | 0.85–0.92 |
| WA → AG | 0.55–0.70 |
| WDC-Co → WDC-Wa | (not in paper) |


In [ ]:
# Quick sanity check against published expectations
import json
from pathlib import Path

EXPECTED = {
    ('wa1', 'ab'): (0.50, 0.65),
    ('wa1', 'da'): (0.85, 0.92),
    ('wa1', 'ag'): (0.55, 0.70),
}

RUNS_DIR = Path(BASE) / 'results' / 'runs' / f'dader-{VARIANT}'
print(f'\n=== Validation against published F1 ranges ===\n')
for src, tgt in PAIRS_TO_RUN:
    m_path = RUNS_DIR / f'{src}__to__{tgt}' / 'metrics.json'
    if not m_path.exists():
        print(f'  {src} → {tgt}: no metrics.json')
        continue
    m = json.loads(m_path.read_text())
    f1 = m.get('test_f1')
    exp_range = EXPECTED.get((src, tgt))
    if f1 is None:
        verdict = '❌ no F1 parsed — check train.log'
    elif exp_range is None:
        verdict = '⚪ not in paper — no comparison'
    elif exp_range[0] - 0.05 <= f1 <= exp_range[1] + 0.05:
        verdict = f'✓ within ±5pp of published [{exp_range[0]}, {exp_range[1]}]'
    else:
        verdict = f'⚠ outside published range [{exp_range[0]}, {exp_range[1]}] — consider Option B/C'
    print(f'  {src} → {tgt}: F1={f1}  {verdict}')


## 7. Push results to git

Adds only metrics.json (train.log is gitignored).


In [ ]:
!git add results/runs/dader-{VARIANT}/
!git status --short


In [ ]:
!git commit -m "DADER runner DADER {VARIANT} baseline on Pair #1 (initial)" || echo 'nothing to commit'
!git push


## 8. After successful Pair #1 — extend to other pairs

If Pair #1 lands within ±5pp of the published WA→AB number, the dependency stack is working. Uncomment the other pairs (the `PAIRS_TO_RUN` config) and re-run cells 9–11. The `SKIP_IF_DONE = True` flag will skip Pair #1.

Total cost for all 4 pairs: ~20 GPU hours.